# Project Scaffolding with uv — A New Hope for Bob

*Bob builds a small Python app, from an empty folder to a distributable package.*

Follow [Chapter 03, Section 01](../../docs/chapter-03/section-01.md) for the reference explanation. Run these cells in order in this lab's Dev Container. All generated projects live in `lab/`; the Jupyter kernel stays on the container's system Python.

## Introduction — One Tool, Many Jobs

> **Bob:** I used to keep a checklist for Python, venv, pip, and packaging. Alice says uv can handle the whole trip. A new hope? I'll believe it when I see the files.

On macOS or Linux, install uv with `curl -LsSf https://astral.sh/uv/install.sh | sh`. On Windows, use PowerShell: `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"`. See [Section 01](../../docs/chapter-03/section-01.md#install-uv) for the copyable commands. The Dev Container has uv installed already; check its version:

In [ ]:
%%bash
uv --version

## Manage Python with uv

 > **Bob:** By Jar Jar's eyes! (Binks, editor's note.) I thought installing an interpreter was a job for the system package manager. Now you're telling me uv can download one itself, without replacing the system Python? That's more than a package manager. The Force is strong with this tool.

Like `nvm` for Node.js and `rustup` for Rust, `uv` can install and select language versions. Use `uv python` to manage interpreters separately from the system installation.

### Installing Python with uv

Install Python 3.12 for the examples below; the notebook kernel stays on the container's system Python:

In [ ]:
%%bash
uv python install 3.12

### Change the Project's Python Pin

Installing an interpreter makes it available; a `.python-version` pin selects it. First, clear only the generated examples and recreate `lab/`, where the shared pin will live:

In [ ]:
from pathlib import Path
import shutil

lab = Path('lab')
if lab.exists():
    shutil.rmtree(lab)
lab.mkdir()
print('Clean lab directory:', lab.resolve())

Pin `lab/` to Python 3.12, then inspect `lab/.python-version` to confirm it contains `3.12`. `--no-project` skips compatibility checks against an existing project; the pin stays local to the lab:

In [ ]:
%%bash
uv --directory lab python pin --no-project 3.12
cat lab/.python-version

## Initialize a Project

### Choose a Project Type

Bob will compare four independent templates. Each `uv init` command inherits Python 3.12 from `lab/.python-version`, so no `--python` flag is needed. Standard templates create their own pin; the bare template continues to use the parent pin.

> **Bob:** First: a quick script, no package to distribute. I don't need a starship for a grocery run.

Create a simple application; `--no-package` deliberately omits a build system:

In [ ]:
%%bash
uv init --app --no-package --vcs none lab/simple-app

Look at its generated files and **complete** `pyproject.toml`. The exact description and uv version may differ from the documentation examples:

In [ ]:
print('Files:', sorted(p.name for p in Path('lab/simple-app').iterdir()))
print(Path('lab/simple-app/pyproject.toml').read_text())

> **Bob:** What if I want my application to be installable, with its own command?

Create a packaged application. This becomes the running example for the rest of the lab:

In [ ]:
%%bash
uv init --app --package --vcs none --author-from none lab/hello-app

Inspect its layout and complete configuration. Find `[project.scripts]`, `[build-system]`, and the package under `src/`:

In [ ]:
print('Files:', sorted(str(p.relative_to('lab/hello-app')) for p in Path('lab/hello-app').rglob('*') if p.is_file()))
print(Path('lab/hello-app/pyproject.toml').read_text())

A library is also a package, but is meant to be **imported** by other projects rather than launched as a command. Create a separate one for comparison:

In [ ]:
%%bash
uv init --lib --vcs none --author-from none lab/hello-library

Compare its complete `pyproject.toml` and source layout with the packaged application:

In [ ]:
print('Files:', sorted(str(p.relative_to('lab/hello-library')) for p in Path('lab/hello-library').rglob('*') if p.is_file()))
print(Path('lab/hello-library/pyproject.toml').read_text())

For full control over the layout, `--bare` starts with only the project file:

In [ ]:
%%bash
uv init --bare --vcs none lab/bare-project

Check that `--bare` did not generate a README or source code:

In [ ]:
print('Files:', sorted(p.name for p in Path('lab/bare-project').iterdir()))
print(Path('lab/bare-project/pyproject.toml').read_text())

Compare the local pin with the project's compatibility requirement:

In [ ]:
print('.python-version:', Path('lab/hello-app/.python-version').read_text().strip())
print('pyproject.toml:\n', Path('lab/hello-app/pyproject.toml').read_text())

`.python-version` selects a Python interpreter for this working copy. `requires-python` in `pyproject.toml` states which versions your code supports when others install the package. They are related, but not the same setting.

## Work in the Project Environment

### Create and Use `.venv`

> **Bob:** No activation script? Let's see where my packages will go.

Prepare the environment; `uv sync` creates `.venv` and `uv.lock` when needed:

In [ ]:
%%bash
uv --directory lab/hello-app sync

Check that both project files appeared. They serve different purposes: `uv.lock` records resolved dependencies; `.venv` is the local environment:

In [ ]:
for name in ('uv.lock', '.venv'):
    print(name, 'exists:', Path('lab/hello-app', name).exists())

### Add Dependencies and Run the Project

Add `rich` as a runtime dependency; uv updates the project file, lockfile, and environment together:

In [ ]:
%%bash
uv --directory lab/hello-app add rich

Let Bob's entry point use the new dependency. Change only its generated `main` function:

> **Bob:** If the terminal prints in color, I'll admit this is more fun than my old checklist.

In [ ]:
entry = Path('lab/hello-app/src/hello_app/__init__.py')
entry.write_text('from rich import print\n\ndef main() -> None:\n    print("[green]Hello from Bob![/green]")\n')
print(entry.read_text())

Run the installed command in the project's `.venv` without manually activating it:

In [ ]:
%%bash
uv --directory lab/hello-app run hello-app

To see exactly how `uv add`, `uv.lock`, and `uv sync` handle dependencies, continue with [Bob's dependency lab](../uv_dependency_resolution/uv_dependency_lab.ipynb) or [Section 02](../../docs/chapter-03/section-02.md).

## Configure Project Tools

### Use `[tool.*]` Tables

Many tools read their own tables in `pyproject.toml`. Add a Ruff setting to the *existing* configuration rather than creating a separate config file:

In [ ]:
project_file = Path('lab/hello-app/pyproject.toml')
project_file.write_text(project_file.read_text() + '\n[tool.ruff]\nline-length = 88\n')
print(project_file.read_text())

`[tool.ruff]` configures Ruff; it does **not** install Ruff. Other tools have their own `[tool.*]` tables. Compare this complete file with the one uv initially generated to see how the configuration grows with the project.

## Build a Package

### Understand the `uv_build` Backend

In the project's `[build-system]` table, `uv_build` names the builder. `uv build` is the command that asks the builder to create distributions. The simple app had no such table.

### Create Distributions

> **Bob:** A wheel and a source archive from one command? That's a much smaller release checklist.

Build the packaged application:

In [ ]:
%%bash
uv --directory lab/hello-app build

Inspect the outputs: the `.whl` is installable; the `.tar.gz` is a source distribution:

In [ ]:
print('Distributions:', sorted(p.name for p in Path('lab/hello-app/dist').iterdir() if p.suffix in {'.whl', '.gz'}))

## Publish a Package

### Upload to a Package Index

`uv publish` uploads the distributions in `dist/` to PyPI; it needs valid package metadata and authentication, such as a PyPI token in `UV_PUBLISH_TOKEN`. For a practice release, use a **TestPyPI** token and `uv publish --publish-url https://test.pypi.org/legacy/`. Bob's `hello-app` is only a local exercise, so this lab stops before uploading anything.

> **Bob:** Publishing is where I'll check the manifest twice. No need to send my training droid to the real galaxy today.

## Manage Related Projects

### Introduce Workspaces

Bob now wants a reusable greeting library next to his app. Create a library **inside** the app's `packages/` directory; uv discovers the existing project and makes the library a workspace member:

In [ ]:
%%bash
uv init --lib --author-from none lab/hello-app/packages/hello-library

Inspect both project files. The root now lists workspace members; the library still owns its own metadata:

In [ ]:
print('Root:\n', Path('lab/hello-app/pyproject.toml').read_text())
print('Member:\n', Path('lab/hello-app/packages/hello-library/pyproject.toml').read_text())

Tell the app to use the local library, rather than looking for a package of the same name on PyPI:

In [ ]:
%%bash
uv --directory lab/hello-app add --workspace hello-library

See the root's dependency and `[tool.uv.sources]` entries. Both projects share the one `uv.lock` at the root:

In [ ]:
print(Path('lab/hello-app/pyproject.toml').read_text())
print('Root lockfile:', Path('lab/hello-app/uv.lock').exists())
print('Member lockfile:', Path('lab/hello-app/packages/hello-library/uv.lock').exists())

## Epilogue — Bob's New Checklist

> **Bob:** I started with one command, and now I can explain where my Python, my project settings, and my built package came from. Alice might have been right.

| Bob's task | uv command or file |
| --- | --- |
| Start an app or library | `uv init` with `--app`, `--package`, or `--lib` |
| Choose local Python | `uv python install`, `uv python pin`, `.python-version` |
| Work on the app | `pyproject.toml`, `uv sync`, `uv add`, `uv run` |
| Configure and distribute | `[tool.*]`, `uv_build`, `uv build`, then `uv publish` when ready |
| Grow into related packages | `[tool.uv.workspace]` and `[tool.uv.sources]` |

Next, [Bob discovers the Force of uv](../uv_dependency_resolution/uv_dependency_lab.ipynb): declaring, locking, and syncing dependencies in depth.